<a href="https://colab.research.google.com/github/Prktriesitout/Concordia-CXR-repo/blob/main/Concordia_CXR_Final_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip show bitsandbytes

Name: bitsandbytes
Version: 0.50.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License-Expression: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, packaging, torch
Required-by: 


In [ ]:
# ============================================================
# MULTI-MODEL CHEST X-RAY ANALYSIS SYSTEM  (Colab T4 edition)
#
# MODEL 1: Google MedGemma 1.5 4B IT   (local, T4 GPU, 4-bit + FP32 compute)
# MODEL 2: OpenAI GPT-6 Astra          (API)
# MODEL 3: OpenAI GPT-6 Astra          (Consensus & Verification)
#
# INPUT : Chest X-ray
# OUTPUT: Consolidated radiology research report
#
# Runtime -> Change runtime type -> T4 GPU
# Colab Secrets needed: HF_TOKEN (optional, else login prompt), OpenAI
# You must accept the MedGemma licence on its Hugging Face page first.
#
# CACHING:
#   - In-session : MedGemma stays in GPU memory. Re-running the cell
#                  (e.g. with a new IMAGE_PATH) skips loading entirely.
#   - Persistent : first run saves the ready-to-use model to Google
#                  Drive. Later sessions copy it back instead of
#                  downloading + quantising again.
# ============================================================


# ============================================================
# 1. INSTALLATION
# ============================================================

#!pip install -q -U "transformers>=4.50.0" accelerate bitsandbytes openai pillow


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import base64
import shutil
from concurrent.futures import ThreadPoolExecutor

import torch
from PIL import Image

from google.colab import userdata
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from openai import OpenAI

from IPython.display import display, Markdown


# ============================================================
# 3. AUTHENTICATION
# ============================================================

def get_secret(name):
    """Return a Colab secret, or None if it is missing / not shared."""
    try:
        return userdata.get(name)
    except Exception:
        return None


# Hugging Face authentication
hf_token = get_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()  # interactive prompt

# OpenAI authentication
openai_key = get_secret("OpenAI")
if not openai_key:
    raise ValueError("OpenAI API key missing from Colab Secrets (name: 'OpenAI').")

client = OpenAI(api_key=openai_key, timeout=300, max_retries=2)

print("Authentication successful.")


# ============================================================
# 4. CONFIGURATION
# ============================================================

IMAGE_PATH = "/content/Bilateral Cystic Lung Changes.png"

MEDGEMMA_MODEL = "google/medgemma-1.5-4b-it"
OPENAI_MODEL = "gpt-6-astra"

# --- Precision settings -------------------------------------
# Gemma-family models produce very large activations. In FP16 they
# overflow to NaN and MedGemma outputs only padding = EMPTY REPORT.
# T4 has no native BF16, so the safe T4 setting is 4-bit weights
# with FP32 compute (~7 GB VRAM, overflow-free).
#
#   "auto"      -> bf16 on A100/L4, 4bit-fp32 on T4   (recommended)
#   "bf16"      -> Ampere+ GPUs only
#   "4bit-fp32" -> T4 safe mode
#   "fp16"      -> fastest on T4 but may give empty output
#   "4bit-fp16" -> smallest, but may give empty output
PRECISION = "auto"

# --- Model cache settings ------------------------------------
CACHE_TO_DRIVE = True   # False = only cache in-session (no Drive)
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/model_cache"
LOCAL_CACHE_ROOT = "/content/model_cache"   # fast local disk

# Tokens dominate MedGemma runtime (~1 token = constant cost).
# 1100 is enough for the full structured report.
MEDGEMMA_MAX_NEW_TOKENS = 1100

# Image sent to OpenAI: downscale very large files to cut upload time.
OPENAI_MAX_SIDE = 2048

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

image = Image.open(IMAGE_PATH).convert("RGB")


def image_to_data_url(img, max_side=OPENAI_MAX_SIDE):
    """Encode a PIL image as a PNG data URL, downscaled if very large."""
    from io import BytesIO
    img = img.copy()
    if max(img.size) > max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    buf = BytesIO()
    img.save(buf, format="PNG", optimize=True)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


image_data_url = image_to_data_url(image)

print("Image loaded:", IMAGE_PATH)
print("Image dimensions:", image.size)


# ============================================================
# 5. COMMON RADIOLOGY INSTRUCTIONS
# ============================================================

RADIOLOGY_PROMPT = """
You are simulating the reporting standards of a highly
experienced senior consultant radiologist with more than
50 years of experience in diagnostic radiology.

Your task is to independently examine the provided
chest X-ray and produce a detailed radiology report.

ANALYSIS REQUIREMENTS:

1. IMAGE QUALITY
- Examine positioning, exposure, rotation and inspiration.
- Identify limitations affecting interpretation.

2. ANATOMICAL EXAMINATION
- Right and left lung fields.
- Pulmonary parenchyma.
- Pleura and costophrenic angles.
- Cardiac silhouette.
- Mediastinum and trachea.
- Diaphragms.
- Visible bones and soft tissues.

3. ABNORMALITY IDENTIFICATION
- Identify opacities, consolidation, infiltrates,
  nodules, effusions or other abnormalities.
- Specify anatomical location and distribution.
- Describe appearance, density and extent.
- Compare bilateral lung fields.

4. INTERPRETATION
- Provide the most likely radiographic interpretation.
- Include differential considerations where appropriate.
- Distinguish observations from clinical interpretations.
- Never invent findings.

5. RELIABILITY
- Never claim perfect diagnostic accuracy.
- Never assume that an abnormality exists.
- Clearly communicate uncertainty.
- Do not infer clinical history from the image.

OUTPUT FORMAT:

# CHEST X-RAY REPORT

## 1. Examination
## 2. Image Quality
## 3. Radiographic Findings
## 4. Abnormality Analysis
## 5. Radiological Impression
## 6. Recommendations

Provide a complete, professional, structured report.

This is an AI-assisted research analysis, not a
clinically validated diagnosis.
"""


# ============================================================
# 6. LOAD MEDGEMMA (cached, optimised for T4)
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Enable a T4 GPU in Colab Runtime settings.")

gpu_name = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)

# (model dtype, 4-bit compute dtype or None for no quantisation)
PRECISION_SETTINGS = {
    "bf16":      (torch.bfloat16, None),
    "fp16":      (torch.float16,  None),
    "4bit-fp32": (torch.float32,  torch.float32),
    "4bit-fp16": (torch.float16,  torch.float16),
}

if PRECISION == "auto":
    precision_tag = "bf16" if major >= 8 else "4bit-fp32"
else:
    precision_tag = PRECISION

if precision_tag not in PRECISION_SETTINGS:
    raise ValueError(f"Unknown PRECISION: {precision_tag}")
if precision_tag == "bf16" and major < 8:
    raise ValueError("bf16 needs an Ampere+ GPU (A100/L4). Use '4bit-fp32' on T4.")

DTYPE, QUANT_COMPUTE_DTYPE = PRECISION_SETTINGS[precision_tag]

# Unique key per setup; each precision gets its own cache folder.
CACHE_KEY = f"{MEDGEMMA_MODEL.split('/')[-1]}-{precision_tag}"
LOCAL_MODEL_DIR = os.path.join(LOCAL_CACHE_ROOT, CACHE_KEY)
DRIVE_MODEL_DIR = os.path.join(DRIVE_CACHE_ROOT, CACHE_KEY)


def is_saved_model(path):
    """True if path holds a complete saved model + processor."""
    if not os.path.isdir(path):
        return False
    files = os.listdir(path)
    has_weights = any(f.endswith(".safetensors") for f in files)
    return has_weights and "config.json" in files and "preprocessor_config.json" in files


def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return True
    except Exception as e:
        print(f"Drive not mounted ({e}). Continuing without persistent cache.")
        return False


def copy_dir(src_dir, dst_dir):
    tmp = dst_dir + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src_dir, tmp)
    shutil.rmtree(dst_dir, ignore_errors=True)
    os.rename(tmp, dst_dir)   # only appears once the copy is complete


def load_medgemma():
    """Load MedGemma: local disk -> Google Drive -> Hugging Face Hub."""
    t0 = time.time()
    use_drive = CACHE_TO_DRIVE and mount_drive()

    # 1) Restore from Drive to fast local disk if needed.
    if not is_saved_model(LOCAL_MODEL_DIR) and use_drive and is_saved_model(DRIVE_MODEL_DIR):
        print("Restoring MedGemma from Google Drive cache...")
        copy_dir(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

    common = dict(
        torch_dtype=DTYPE,
        device_map="cuda:0",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )

    if is_saved_model(LOCAL_MODEL_DIR):
        # 2) Load the cached copy (already quantised if 4-bit).
        print("Loading MedGemma from local cache:", LOCAL_MODEL_DIR)
        processor = AutoProcessor.from_pretrained(LOCAL_MODEL_DIR)
        model = AutoModelForImageTextToText.from_pretrained(LOCAL_MODEL_DIR, **common)
    else:
        # 3) First run: download from the Hub, then save the cache.
        print("First run: downloading MedGemma from Hugging Face...")
        if QUANT_COMPUTE_DTYPE is not None:
            common["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=QUANT_COMPUTE_DTYPE,
                bnb_4bit_use_double_quant=True,
            )
        processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL)
        model = AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL, **common)

        try:
            print("Saving model to local cache...")
            model.save_pretrained(LOCAL_MODEL_DIR, safe_serialization=True)
            processor.save_pretrained(LOCAL_MODEL_DIR)
            if use_drive:
                print("Saving model to Google Drive cache (one-time)...")
                os.makedirs(DRIVE_CACHE_ROOT, exist_ok=True)
                copy_dir(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
                print("Drive cache saved:", DRIVE_MODEL_DIR)
        except Exception as e:
            print(f"Warning: could not save cache ({e}). Model still usable.")

    model.eval()
    print(f"MedGemma ready in {time.time() - t0:.1f}s")
    return processor, model


# In-session cache: skip loading if this exact model is already in memory.
if globals().get("_MEDGEMMA_CACHE_KEY") == CACHE_KEY and "model" in globals():
    print(f"\nMedGemma already loaded ({CACHE_KEY}) - using in-memory model.")
else:
    # Free any previously loaded variant before loading a new one.
    for _name in ("model", "processor"):
        globals().pop(_name, None)
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\nLoading MedGemma on {gpu_name} ({precision_tag})...")
    processor, model = load_medgemma()
    _MEDGEMMA_CACHE_KEY = CACHE_KEY

print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def build_inputs(user_text, system_text=None):
    messages = []
    if system_text:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_text}]})
    messages.append({
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text},
        ],
    })
    return processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=DTYPE)   # casts pixel_values only; ids stay int


def medgemma_self_test():
    """Short generation to catch NaN/overflow before the long report."""
    inputs = build_inputs("Describe this image in one sentence.")
    n = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    new = out[0][n:]
    text = processor.decode(new, skip_special_tokens=True).strip()
    pad_id = processor.tokenizer.pad_token_id
    if not text:
        only_pad = bool((new == pad_id).all())
        raise RuntimeError(
            "MedGemma self-test produced no text "
            f"({'only padding tokens -> numeric overflow' if only_pad else 'raw ids: ' + str(new[:10].tolist())}). "
            f"Current PRECISION = '{precision_tag}'. On a T4 use PRECISION = '4bit-fp32'."
        )
    print("Self-test OK:", text[:120])


if not globals().get("_MEDGEMMA_TESTED") == CACHE_KEY:
    print("Running MedGemma self-test...")
    medgemma_self_test()
    _MEDGEMMA_TESTED = CACHE_KEY


# ============================================================
# 7. MODEL 1: MEDGEMMA ANALYSIS
# ============================================================

def analyze_with_medgemma():

    print("\n[MODEL 1] MedGemma analysis started...")
    start = time.time()

    inputs = build_inputs(
        "Independently analyse this chest X-ray.",
        system_text=RADIOLOGY_PROMPT,
    )

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MEDGEMMA_MAX_NEW_TOKENS,
            do_sample=False,          # greedy: fastest & deterministic
            use_cache=True,
            repetition_penalty=1.05,  # stops 4B model looping
        )

    new_tokens = output_ids[0][input_len:]
    report = processor.decode(new_tokens, skip_special_tokens=True).strip()

    del inputs, output_ids
    torch.cuda.empty_cache()

    if not report:
        raise RuntimeError(
            f"MedGemma returned an empty report (PRECISION = '{precision_tag}'). "
            "On a T4 set PRECISION = '4bit-fp32' and re-run."
        )

    elapsed = time.time() - start
    print(f"[MODEL 1] MedGemma completed in {elapsed:.1f}s "
          f"({len(new_tokens) / elapsed:.1f} tokens/s).")

    return report


# ============================================================
# 8. MODEL 2: GPT-6 ASTRA INDEPENDENT ANALYSIS
# ============================================================

def analyze_with_openai():

    print("\n[MODEL 2] GPT-6 Astra analysis started...")
    start = time.time()

    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=RADIOLOGY_PROMPT,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Independently examine the original "
                            "chest X-ray and generate a complete "
                            "radiology report. Do not assume any "
                            "previous model's interpretation."
                        ),
                    },
                    {
                        "type": "input_image",
                        "image_url": image_data_url,
                        "detail": "high",
                    },
                ],
            }
        ],
        max_output_tokens=1800,
    )

    report = (response.output_text or "").strip()

    if not report:
        raise RuntimeError("OpenAI returned an empty report.")

    print(f"[MODEL 2] GPT-6 Astra completed in {time.time() - start:.1f}s.")

    return report


# ============================================================
# 9. PARALLEL EXECUTION OF MODEL 1 AND MODEL 2
# ============================================================
# MedGemma runs on the GPU while the OpenAI request runs over
# the network, so total time ~= the slower of the two.

print("\n" + "=" * 65)
print("STARTING INDEPENDENT MODEL ANALYSIS")
print("=" * 65)

with ThreadPoolExecutor(max_workers=2) as executor:
    future_medgemma = executor.submit(analyze_with_medgemma)
    future_openai = executor.submit(analyze_with_openai)

    # Collect each result separately so one failure doesn't hide the other.
    try:
        medgemma_report = future_medgemma.result()
    except Exception as e:
        print(f"\n[MODEL 1] FAILED: {e}")
        medgemma_report = f"*MedGemma report unavailable: {e}*"

    try:
        openai_report = future_openai.result()
    except Exception as e:
        print(f"\n[MODEL 2] FAILED: {e}")
        openai_report = f"*GPT-6 Astra report unavailable: {e}*"

print("\nIndependent analysis stage finished.")


# ============================================================
# 10. DISPLAY INDIVIDUAL MODEL REPORTS
# ============================================================

print("\nMODEL 1: MEDGEMMA REPORT")
display(Markdown(medgemma_report))

print("\nMODEL 2: GPT-6 ASTRA REPORT")
display(Markdown(openai_report))


# ============================================================
# 11. MODEL 3: GPT-6 ASTRA VERIFICATION ENGINE
# ============================================================

VERIFICATION_PROMPT = """
ROLE:

You are the final radiology report reviewer in a
multi-model AI-assisted medical imaging research system.

You are performing a second-pass review using the same
GPT-6 Astra model that produced one of the initial reports.

You have received:
1. The original chest X-ray.
2. An independent report generated by Google MedGemma.
3. An independent report generated by GPT-6 Astra.

YOUR PRIMARY RESPONSIBILITY:

Independently re-examine the ORIGINAL X-ray, compare both
reports, identify agreements and disagreements, and produce
one carefully consolidated final report.

IMPORTANT:
Do not blindly trust either model.
Do not assume agreement means correctness.
Do not automatically prefer one model over the other.
Do not treat the reports as ground truth.
If one report is marked unavailable, say so and rely on the
image and the remaining report.

VERIFICATION PROCESS:

1. INDEPENDENT IMAGE REVIEW
Examine the original image before relying on either report.

2. AGREEMENT ANALYSIS
Identify findings independently reported by both models.

3. DISAGREEMENT ANALYSIS
Identify findings where the models differ regarding:
- Anatomical location.
- Type of abnormality.
- Extent or severity.
- Presence or absence of a finding.
- Interpretation and differential diagnosis.

4. EVIDENCE-BASED ADJUDICATION
For each disagreement:
- Re-examine the corresponding image region.
- Determine whether the finding is visibly supported.
- Reject unsupported claims.
- Preserve uncertainty where the image is inconclusive.
- If the image cannot resolve a disagreement, state so.

5. FINAL REPORT
Generate a consolidated report based on the original image,
using both reports as supporting evidence rather than truth.

OUTPUT FORMAT:

# MULTI-MODEL CHEST X-RAY ANALYSIS

## 1. Model Agreement
Summarize findings supported by both models.

## 2. Model Disagreements
Describe conflicting observations and interpretations.

## 3. Verification Results
Explain which findings are visually supported,
unsupported or indeterminate.

## 4. FINAL CONSOLIDATED RADIOLOGY REPORT

### Examination
### Image Quality
### Radiographic Findings
### Abnormality Analysis
### Radiological Impression
### Recommendations

## 5. Limitations
Mention uncertainties, technical limitations and
the limitations of AI-based interpretation.

## 6. Final Summary
Provide a concise summary of the most important
radiographic observations.

ADDITIONAL REQUIREMENTS:

- The original X-ray is the primary evidence.
- Never fabricate image findings.
- Never claim that the report is error-free.
- Never present model agreement as proof of accuracy.
- Do not provide a numerical confidence score unless
  it is supported by a validated calibration method.
- Do not reveal private internal reasoning.
- Provide concise, evidence-based explanations.
- Use professional radiology terminology.
- Maintain consistent findings and conclusions.

This is an AI-assisted research report and must be
reviewed by a qualified radiologist before clinical use.
"""


# ============================================================
# 12. PREPARE VERIFICATION INPUT
# ============================================================

verification_input = f"""
You are provided with two independently generated
radiology reports.

==============================
MEDGEMMA REPORT (MODEL 1)
==============================

{medgemma_report}


==============================
GPT-6 ASTRA REPORT (MODEL 2)
==============================

{openai_report}


==============================
YOUR TASK
==============================

Independently inspect the attached original X-ray.

Compare both reports, identify agreements and disagreements,
verify the observations against the image, and produce
the final consolidated radiology report.

Do not simply merge or repeat the two reports.
"""


# ============================================================
# 13. STREAM THE FINAL VERIFICATION REPORT
# ============================================================

print("\n" + "=" * 65)
print("MODEL 3: FINAL VERIFICATION STARTED")
print("=" * 65)
print("GPT-6 Astra is reviewing both reports and the original image.")

final_display = display(
    Markdown("*Generating consolidated radiology report...*"),
    display_id=True,
)

final_chunks = []
final_report = ""
last_render = 0.0

try:
    stream = client.responses.create(
        model=OPENAI_MODEL,
        instructions=VERIFICATION_PROMPT,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": verification_input},
                    {
                        "type": "input_image",
                        "image_url": image_data_url,
                        "detail": "high",
                    },
                ],
            }
        ],
        max_output_tokens=3000,
        stream=True,
    )

    for event in stream:
        if event.type == "response.output_text.delta":
            final_chunks.append(event.delta)

            # Re-render at most ~3x per second (cheap for the notebook).
            now = time.time()
            if now - last_render > 0.3:
                final_display.update(Markdown("".join(final_chunks)))
                last_render = now

        elif event.type in ("response.failed", "error"):
            raise RuntimeError(f"Streaming error: {event}")

    final_report = "".join(final_chunks).strip()

    if not final_report:
        raise RuntimeError("Final verification returned an empty report.")

    final_display.update(Markdown(final_report))
    print("\nFinal verification completed.")

except Exception as e:
    final_display.update(Markdown(f"**Verification Error:** {e}"))
    final_report = "".join(final_chunks) or f"Verification failed: {e}"
    print(f"\n[MODEL 3] FAILED: {e}")


# ============================================================
# 14. SAVE ALL REPORTS
# ============================================================

output_directory = "/content/Xray_Analysis_Reports"
os.makedirs(output_directory, exist_ok=True)

reports = {
    "01_MedGemma_Report.md": medgemma_report,
    "02_GPT6_Astra_Report.md": openai_report,
    "03_Final_Verified_Report.md": final_report,
}

for filename, content in reports.items():
    file_path = os.path.join(output_directory, filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Saved:", file_path)


# ============================================================
# 15. FINAL STATUS
# ============================================================

def status(report):
    return "[FAILED]" if report.lstrip().startswith(("*", "Verification failed")) else "[COMPLETED]"

print("\n" + "=" * 65)
print("MULTI-MODEL ANALYSIS COMPLETED")
print("=" * 65)

print(f"""
MODEL 1 : MedGemma 1.5 4B       {status(medgemma_report)}
MODEL 2 : GPT-6 Astra           {status(openai_report)}
MODEL 3 : GPT-6 Astra Reviewer  {status(final_report)}

OUTPUTS:
1. MedGemma independent report
2. GPT-6 Astra independent report
3. Consolidated verification report

Location:
{output_directory}
""")

print("All reports have been saved.")

Authentication successful.
Image loaded: /content/Bilateral Cystic Lung Changes.png
Image dimensions: (1312, 1199)

MedGemma already loaded (medgemma-1.5-4b-it-4bit-fp32) - using in-memory model.
GPU memory used: 4.63 GB

STARTING INDEPENDENT MODEL ANALYSIS

[MODEL 1] MedGemma analysis started...

[MODEL 2] GPT-6 Astra analysis started...
[MODEL 2] GPT-6 Astra completed in 18.6s.
[MODEL 1] MedGemma completed in 95.9s (11.5 tokens/s).

Independent analysis stage finished.

MODEL 1: MEDGEMMA REPORT


<unused94>thought
The user wants me to act as a senior consultant radiologist and provide a detailed chest X-ray report for the given image.

Here's a breakdown of the steps I need to follow:

1.  **Understand the Persona:** Adopt the voice and perspective of a highly experienced senior consultant radiologist (50+ years). This means using precise terminology, focusing on subtle details, acknowledging limitations, and expressing cautious interpretations.
2.  **Analyze the Image:** Carefully examine the provided chest X-ray.
    *   **Image Quality:** Assess positioning (PA), exposure (appears adequate, though maybe slightly low), rotation (minimal), inspiration (good), and any artifacts. Note any limitations (e.g., patient cooperation, potential for suboptimal technique).
    *   **Anatomical Examination:** Systematically review all structures:
        *   Right/Left Lung Fields: Observe overall appearance, symmetry, presence of opacities, infiltrates, etc.
        *   Pulmonary Parenchyma: Look for patterns like interstitial thickening, emphysema, nodules, masses.
        *   Pleura/Costophrenic Angles: Check for blunting, effusions, pneumothorax.
        *   Cardiac Silhouette: Assess size, shape, and contour.
        *   Mediastinum/Trachea: Evaluate position, width, and tracheal deviation.
        *   Diaphragms: Check for elevation, flattening, or focal lesions.
        *   Visible Bones/Soft Tissues: Look for fractures, masses, or other abnormalities.
    *   **Abnormality Identification:** Pinpoint specific findings.
        *   Opacities/Consolidation/Infiltrates: Where are they located? What do they look like (e.g., patchy, diffuse, nodular)? Are they unilateral or bilateral?
        *   Nodules/Masses: Size, shape, density, location.
        *   Effusions: Location, size, density.
        *   Other: Ectasia, atelectasis, etc.
    *   **Interpretation:** Formulate the most likely radiographic interpretation based on the findings. Consider differential diagnoses. Distinguish between observed findings and clinical interpretations. Avoid inventing findings.
    *   **Reliability:** Acknowledge limitations and uncertainties. State that this is not a definitive diagnosis.
3.  **Structure the Report:** Follow the specified output format:
    *   `# CHEST X-RAY REPORT`
    *   `## 1. Examination`
    *   `## 2. Image Quality`
    *   `## 3. Radiographic Findings`
    *   `## 4. Abnormality Analysis`
    *   `## 5. Radiological Impression`
    *   `## 6. Recommendations`
4.  **Write the Report:** Draft the report using the persona's voice and adhering to the structure and requirements. Use precise radiological terms. Be cautious and avoid definitive statements where uncertainty exists.

**Detailed Analysis of the Image:**

*   **Image Quality:**
    *   Positioning: PA view.
    *   Exposure: Appears adequate, but perhaps slightly low, potentially leading to some increased density in the lung vasculature.
    *   Rotation: Minimal.
    *   Inspiration: Good.
    *   Artifacts: None apparent.
    *   Limitations: The image quality is good, but the patient's cooperation for optimal inspiration might be limited.
*   **Anatomical Examination:**
    *   Lung Fields: Both lung fields appear relatively clear, but there is significant prominence of the pulmonary vasculature, suggesting possible mild pulmonary vascular congestion or simply a normal finding in this patient. There is a noticeable increase in interstitial markings, particularly in the upper lobes.
    *   Parenchyma: There is a marked increase in interstitial markings, appearing as fine lines, suggestive of interstitial lung disease or possibly related to the vascular prominence. No obvious consolidation, nodules, or masses are identified.
    *   Pleura/Costophrenic Angles: The costophrenic angles are sharp bilaterally. No evidence of pleural effusion.
    *   Cardiac Silhouette: The cardiac silhouette appears within normal limits in size and contour.
    *   Mediastinum/Trachea: The mediastinum appears unremarkable. The trachea is midline.
    *   Diaphragms: The diaphragms are well-defined.
    *   Bones/Soft Tissues: The visualized bony structures appear unremarkable.
*   **Abnormality Identification:**
    *   Primary Finding: Increased interstitial markings, particularly prominent in the upper lung zones. This could represent interstitial lung disease (ILD) or potentially related to pulmonary vascular changes.
    *   Other Findings: Prominent pulmonary vasculature.
*   **Interpretation:**
    *   The most striking finding is the increased interstitial markings, especially in the upper lobes. This pattern is suggestive of an underlying interstitial lung process.
    *   Differential Considerations: Could be related to various ILDs (e.g., idiopathic pulmonary fibrosis, nonspecific interstitial pneumonia, hypersensitivity pneumonitis), or potentially secondary to pulmonary vascular congestion (though less likely without other signs of heart failure). Other


MODEL 2: GPT-6 ASTRA REPORT


# CHEST X-RAY REPORT

## 1. Examination
Single frontal chest radiograph, apparently obtained in the PA projection. No lateral view or prior studies are available for comparison.

## 2. Image Quality
Adequate inspiration and exposure, with minimal rotation. Both lungs and costophrenic angles are included. Assessment of the distribution and nature of the extensive cystic pulmonary abnormalities is limited by the single projection.

## 3. Radiographic Findings
- **Lungs and airways:** Extensive bilateral ring-shaped and branching opacities surrounding multiple air-filled cystic spaces of varying size. These changes predominantly involve the middle and lower lung zones, greater on the right, with additional right upper-zone involvement. Associated coarse linear opacities, bronchial wall thickening, and architectural distortion are present. Several cystic spaces have relatively thick walls. No definite air–fluid levels are identified.
- No focal dense air-space consolidation or discrete solid pulmonary mass is evident. The extensive background abnormalities limit assessment for small nodules or subtle superimposed infiltrates.
- **Pleura:** No pleural effusion or pneumothorax. Costophrenic angles are clear.
- **Cardiomediastinal structures:** Cardiac silhouette is not enlarged. No mediastinal widening. Trachea is approximately midline.
- **Diaphragms:** Maintained contours without focal elevation.
- **Bones and soft tissues:** No acute abnormality identified in the visualized structures.

## 4. Abnormality Analysis
The combination of numerous thick-walled ring shadows, branching airway-related opacities, and mid-to-lower-zone predominance strongly favors **extensive cystic/saccular bronchiectasis**, with associated chronic parenchymal scarring.

Other cystic or bullous pulmonary abnormalities are less favored by the prominent walls and airway-related pattern but cannot be definitively distinguished on radiography alone. The underlying cause cannot be established from this image. No definite focal radiographic evidence of superimposed pneumonia is seen; active airway infection cannot be excluded by chest radiography.

## 5. Radiological Impression
1. **Extensive bilateral cystic bronchiectatic changes**, predominantly in the middle and lower lung zones, right greater than left, with associated chronic architectural distortion.
2. No focal dense consolidation, pleural effusion, or pneumothorax.

## 6. Recommendations
- Compare with previous chest radiographs or CT studies to establish stability.
- If not previously characterized, thin-section chest CT is recommended to confirm bronchiectasis and define its extent and distribution.
- Correlate with respiratory symptoms and, if infection is suspected, appropriate clinical and sputum assessment.

*This is an AI-assisted research analysis, not a clinically validated diagnosis.*


MODEL 3: FINAL VERIFICATION STARTED
GPT-6 Astra is reviewing both reports and the original image.


# MULTI-MODEL CHEST X-RAY ANALYSIS

## 1. Model Agreement
Both reports describe:
- Bilateral abnormal linear pulmonary markings, although their characterization differs substantially.
- A nonenlarged cardiac silhouette and approximately midline trachea.
- No focal dense consolidation or pleural effusion.
- No obvious acute abnormality of the visualized bones.

Agreement alone does not establish accuracy; these observations were assessed against the supplied image.

## 2. Model Disagreements
- **Dominant abnormality:** MedGemma describes predominantly interstitial markings and vascular prominence. GPT-6 Astra describes extensive cystic spaces, ring shadows, and bronchial wall thickening.
- **Distribution:** MedGemma emphasizes the upper lung zones; GPT-6 Astra emphasizes the middle and lower zones, greater on the right.
- **Interpretation:** MedGemma suggests interstitial lung disease or vascular congestion. GPT-6 Astra favors extensive cystic/saccular bronchiectasis.
- **Extent:** MedGemma’s description of relatively clear lungs understates the substantial bilateral abnormalities described by GPT-6 Astra.

The supplied MedGemma text is incomplete and ends during its differential discussion.

## 3. Verification Results
- **Visually supported:** Numerous bilateral air-filled, ring-shaped spaces with conspicuous walls, associated coarse linear/branching opacities, and architectural distortion. These abnormalities are greatest in the middle and lower zones, more extensive on the right, with additional upper-zone involvement.
- **Favored interpretation:** The morphology strongly favors extensive cystic bronchiectasis. CT is needed to confirm airway continuity and distinguish bronchiectatic spaces from other cystic or bullous abnormalities.
- **Not supported as the dominant explanation:** Pulmonary vascular congestion or a nonspecific upper-zone interstitial process does not adequately explain the prominent cystic/ring-shadow pattern. There is no convincing radiographic edema pattern.
- **Supported negative findings:** No focal dense air-space consolidation, pleural effusion, or pneumothorax. Heart size is normal.
- **Indeterminate:** Underlying cause, temporal stability, and the presence of active airway infection cannot be established from this image alone.

## 4. FINAL CONSOLIDATED RADIOLOGY REPORT

### Examination
Single frontal chest radiograph, apparently PA. No lateral view or comparison imaging provided.

### Image Quality
Adequate inspiration and exposure with minimal rotation. Extensive overlapping pulmonary abnormalities limit assessment for small nodules and subtle superimposed infiltrates.

### Radiographic Findings
Extensive bilateral cystic/ring-shaped pulmonary lucencies of varying size, many with conspicuous walls, accompanied by coarse linear and branching opacities and architectural distortion. Changes predominantly involve the middle and lower lung zones, greater on the right, with additional upper-zone involvement. No definite air–fluid levels are identified.

No focal dense air-space consolidation, pleural effusion, or pneumothorax.

Cardiomediastinal silhouette is not enlarged. Trachea is approximately midline. No acute osseous abnormality is evident.

### Abnormality Analysis
The extensive ring shadows and airway-pattern opacities strongly favor cystic/saccular bronchiectasis with associated chronic parenchymal remodeling. Other cystic or bullous lung abnormalities cannot be definitively excluded on radiography. The underlying etiology is not established.

### Radiological Impression
1. Extensive bilateral cystic bronchiectatic changes, predominantly in the middle and lower lung zones, right greater than left.
2. No focal dense consolidation, pleural effusion, or pneumothorax. Superimposed airway infection cannot be excluded radiographically.

### Recommendations
- Compare with prior radiographs or chest CT to assess stability.
- If not previously characterized, obtain thin-section chest CT to confirm the diagnosis and define its extent.
- Correlate with respiratory symptoms; consider sputum assessment if infection is suspected.

## 5. Limitations
Assessment is limited to a single supplied frontal image without clinical history or prior studies. Radiography cannot reliably determine the cause of these changes or exclude subtle superimposed infection. AI-based interpretation may miss or misclassify findings.

**This AI-assisted research report must be reviewed by a qualified radiologist before clinical use.**

## 6. Final Summary
Extensive bilateral cystic/ring-shadow abnormalities, greatest in the middle and lower lungs, strongly favor cystic bronchiectasis rather than vascular congestion or nonspecific interstitial disease. Thin-section CT is appropriate if these findings have not already been characterized.


Final verification completed.
Saved: /content/Xray_Analysis_Reports/01_MedGemma_Report.md
Saved: /content/Xray_Analysis_Reports/02_GPT6_Astra_Report.md
Saved: /content/Xray_Analysis_Reports/03_Final_Verified_Report.md

MULTI-MODEL ANALYSIS COMPLETED

MODEL 1 : MedGemma 1.5 4B       [COMPLETED]
MODEL 2 : GPT-6 Astra           [COMPLETED]
MODEL 3 : GPT-6 Astra Reviewer  [COMPLETED]

OUTPUTS:
1. MedGemma independent report
2. GPT-6 Astra independent report
3. Consolidated verification report

Location:
/content/Xray_Analysis_Reports

All reports have been saved.


#FINAL CODE

In [ ]:
# ============================================================
# MULTI-MODEL CHEST X-RAY ANALYSIS  -  GRADIO APP (Colab T4)
#
# MODEL 1: Google MedGemma 1.5 4B IT   (local GPU, cached)
# MODEL 2: OpenAI GPT-6 Astra          (API)
# MODEL 3: OpenAI GPT-6 Astra          (Consensus & Verification)
#
# Runtime -> Change runtime type -> T4 GPU
# Colab Secrets: HF_TOKEN (optional, else login prompt), OpenAI
# Accept the MedGemma licence on its Hugging Face page first.
#
# Run this cell once. MedGemma loads (from cache when available),
# then a public Gradio link appears below.
# ============================================================


# ============================================================
# 1. INSTALLATION  (run once per fresh runtime)
# ============================================================

# !pip install -q -U "transformers>=4.50.0" accelerate bitsandbytes openai pillow "gradio>=5"


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import base64
import shutil
import threading
from io import BytesIO
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import torch
from PIL import Image

import gradio as gr
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
from openai import OpenAI


# ============================================================
# 3. AUTHENTICATION
# ============================================================

def get_secret(name):
    """Return a Colab secret, or None if it is missing / not shared."""
    try:
        return userdata.get(name)
    except Exception:
        return None


hf_token = get_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()

openai_key = get_secret("OpenAI")
if not openai_key:
    raise ValueError("OpenAI API key missing from Colab Secrets (name: 'OpenAI').")

client = OpenAI(api_key=openai_key, timeout=300, max_retries=2)
print("Authentication successful.")


# ============================================================
# 4. CONFIGURATION
# ============================================================

MEDGEMMA_MODEL = "google/medgemma-1.5-4b-it"
OPENAI_MODEL = "gpt-6-astra"

# "auto" -> bf16 on A100/L4, 4bit-fp32 on T4 (overflow-safe)
# other options: "bf16", "4bit-fp32", "fp16", "4bit-fp16"
PRECISION = "auto"

CACHE_TO_DRIVE = True
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/model_cache"
LOCAL_CACHE_ROOT = "/content/model_cache"

MEDGEMMA_MAX_NEW_TOKENS = 1100
OPENAI_MAX_SIDE = 2048
OUTPUT_ROOT = "/content/Xray_Analysis_Reports"


# ============================================================
# 5. PROMPTS
# ============================================================

RADIOLOGY_PROMPT = """
You are simulating the reporting standards of a highly
experienced senior consultant radiologist with more than
50 years of experience in diagnostic radiology.

Your task is to independently examine the provided
chest X-ray and produce a detailed radiology report.

ANALYSIS REQUIREMENTS:

1. IMAGE QUALITY
- Examine positioning, exposure, rotation and inspiration.
- Identify limitations affecting interpretation.

2. ANATOMICAL EXAMINATION
- Right and left lung fields.
- Pulmonary parenchyma.
- Pleura and costophrenic angles.
- Cardiac silhouette.
- Mediastinum and trachea.
- Diaphragms.
- Visible bones and soft tissues.

3. ABNORMALITY IDENTIFICATION
- Identify opacities, consolidation, infiltrates,
  nodules, effusions or other abnormalities.
- Specify anatomical location and distribution.
- Describe appearance, density and extent.
- Compare bilateral lung fields.

4. INTERPRETATION
- Provide the most likely radiographic interpretation.
- Include differential considerations where appropriate.
- Distinguish observations from clinical interpretations.
- Never invent findings.

5. RELIABILITY
- Never claim perfect diagnostic accuracy.
- Never assume that an abnormality exists.
- Clearly communicate uncertainty.
- Do not infer clinical history from the image.

OUTPUT FORMAT:

# CHEST X-RAY REPORT

## 1. Examination
## 2. Image Quality
## 3. Radiographic Findings
## 4. Abnormality Analysis
## 5. Radiological Impression
## 6. Recommendations

Provide a complete, professional, structured report.

This is an AI-assisted research analysis, not a
clinically validated diagnosis.
"""

VERIFICATION_PROMPT = """
ROLE:

You are the final radiology report reviewer in a
multi-model AI-assisted medical imaging research system.

You are performing a second-pass review using the same
GPT-6 Astra model that produced one of the initial reports.

You have received:
1. The original chest X-ray.
2. An independent report generated by Google MedGemma.
3. An independent report generated by GPT-6 Astra.

YOUR PRIMARY RESPONSIBILITY:

Independently re-examine the ORIGINAL X-ray, compare both
reports, identify agreements and disagreements, and produce
one carefully consolidated final report.

IMPORTANT:
Do not blindly trust either model.
Do not assume agreement means correctness.
Do not automatically prefer one model over the other.
Do not treat the reports as ground truth.
If one report is marked unavailable, say so and rely on the
image and the remaining report.

VERIFICATION PROCESS:

1. INDEPENDENT IMAGE REVIEW
Examine the original image before relying on either report.

2. AGREEMENT ANALYSIS
Identify findings independently reported by both models.

3. DISAGREEMENT ANALYSIS
Identify findings where the models differ regarding:
- Anatomical location.
- Type of abnormality.
- Extent or severity.
- Presence or absence of a finding.
- Interpretation and differential diagnosis.

4. EVIDENCE-BASED ADJUDICATION
For each disagreement:
- Re-examine the corresponding image region.
- Determine whether the finding is visibly supported.
- Reject unsupported claims.
- Preserve uncertainty where the image is inconclusive.
- If the image cannot resolve a disagreement, state so.

5. FINAL REPORT
Generate a consolidated report based on the original image,
using both reports as supporting evidence rather than truth.

OUTPUT FORMAT:

# MULTI-MODEL CHEST X-RAY ANALYSIS

## 1. Model Agreement
Summarize findings supported by both models.

## 2. Model Disagreements
Describe conflicting observations and interpretations.

## 3. Verification Results
Explain which findings are visually supported,
unsupported or indeterminate.

## 4. FINAL CONSOLIDATED RADIOLOGY REPORT

### Examination
### Image Quality
### Radiographic Findings
### Abnormality Analysis
### Radiological Impression
### Recommendations

## 5. Limitations
Mention uncertainties, technical limitations and
the limitations of AI-based interpretation.

## 6. Final Summary
Provide a concise summary of the most important
radiographic observations.

ADDITIONAL REQUIREMENTS:

- The original X-ray is the primary evidence.
- Never fabricate image findings.
- Never claim that the report is error-free.
- Never present model agreement as proof of accuracy.
- Do not provide a numerical confidence score unless
  it is supported by a validated calibration method.
- Do not reveal private internal reasoning.
- Provide concise, evidence-based explanations.
- Use professional radiology terminology.
- Maintain consistent findings and conclusions.

This is an AI-assisted research report and must be
reviewed by a qualified radiologist before clinical use.
"""


def build_verification_input(medgemma_report, openai_report):
    return f"""
You are provided with two independently generated
radiology reports.

==============================
MEDGEMMA REPORT (MODEL 1)
==============================

{medgemma_report}


==============================
GPT-6 ASTRA REPORT (MODEL 2)
==============================

{openai_report}


==============================
YOUR TASK
==============================

Independently inspect the attached original X-ray.

Compare both reports, identify agreements and disagreements,
verify the observations against the image, and produce
the final consolidated radiology report.

Do not simply merge or repeat the two reports.
"""


# ============================================================
# 6. LOAD MEDGEMMA (cached, T4-safe precision)
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Enable a T4 GPU in Colab Runtime settings.")

gpu_name = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)

PRECISION_SETTINGS = {
    "bf16":      (torch.bfloat16, None),
    "fp16":      (torch.float16,  None),
    "4bit-fp32": (torch.float32,  torch.float32),
    "4bit-fp16": (torch.float16,  torch.float16),
}

precision_tag = ("bf16" if major >= 8 else "4bit-fp32") if PRECISION == "auto" else PRECISION
if precision_tag not in PRECISION_SETTINGS:
    raise ValueError(f"Unknown PRECISION: {precision_tag}")
if precision_tag == "bf16" and major < 8:
    raise ValueError("bf16 needs an Ampere+ GPU (A100/L4). Use '4bit-fp32' on T4.")

DTYPE, QUANT_COMPUTE_DTYPE = PRECISION_SETTINGS[precision_tag]

CACHE_KEY = f"{MEDGEMMA_MODEL.split('/')[-1]}-{precision_tag}"
LOCAL_MODEL_DIR = os.path.join(LOCAL_CACHE_ROOT, CACHE_KEY)
DRIVE_MODEL_DIR = os.path.join(DRIVE_CACHE_ROOT, CACHE_KEY)


def is_saved_model(path):
    if not os.path.isdir(path):
        return False
    files = os.listdir(path)
    return (
        any(f.endswith(".safetensors") for f in files)
        and "config.json" in files
        and "preprocessor_config.json" in files
    )


def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return True
    except Exception as e:
        print(f"Drive not mounted ({e}). Continuing without persistent cache.")
        return False


def copy_dir(src_dir, dst_dir):
    tmp = dst_dir + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src_dir, tmp)
    shutil.rmtree(dst_dir, ignore_errors=True)
    os.rename(tmp, dst_dir)


def load_medgemma():
    """Load MedGemma: local disk -> Google Drive -> Hugging Face Hub."""
    t0 = time.time()
    use_drive = CACHE_TO_DRIVE and mount_drive()

    if not is_saved_model(LOCAL_MODEL_DIR) and use_drive and is_saved_model(DRIVE_MODEL_DIR):
        print("Restoring MedGemma from Google Drive cache...")
        copy_dir(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

    common = dict(
        torch_dtype=DTYPE,
        device_map="cuda:0",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )

    if is_saved_model(LOCAL_MODEL_DIR):
        print("Loading MedGemma from local cache:", LOCAL_MODEL_DIR)
        proc = AutoProcessor.from_pretrained(LOCAL_MODEL_DIR)
        mdl = AutoModelForImageTextToText.from_pretrained(LOCAL_MODEL_DIR, **common)
    else:
        print("First run: downloading MedGemma from Hugging Face...")
        if QUANT_COMPUTE_DTYPE is not None:
            common["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=QUANT_COMPUTE_DTYPE,
                bnb_4bit_use_double_quant=True,
            )
        proc = AutoProcessor.from_pretrained(MEDGEMMA_MODEL)
        mdl = AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL, **common)

        try:
            print("Saving model to local cache...")
            mdl.save_pretrained(LOCAL_MODEL_DIR, safe_serialization=True)
            proc.save_pretrained(LOCAL_MODEL_DIR)
            if use_drive:
                print("Saving model to Google Drive cache (one-time)...")
                os.makedirs(DRIVE_CACHE_ROOT, exist_ok=True)
                copy_dir(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
        except Exception as e:
            print(f"Warning: could not save cache ({e}). Model still usable.")

    mdl.eval()
    print(f"MedGemma ready in {time.time() - t0:.1f}s")
    return proc, mdl


if globals().get("_MEDGEMMA_CACHE_KEY") == CACHE_KEY and "model" in globals():
    print(f"MedGemma already loaded ({CACHE_KEY}) - using in-memory model.")
else:
    for _name in ("model", "processor"):
        globals().pop(_name, None)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading MedGemma on {gpu_name} ({precision_tag})...")
    processor, model = load_medgemma()
    _MEDGEMMA_CACHE_KEY = CACHE_KEY

print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def build_inputs(img, user_text, system_text=None):
    messages = []
    if system_text:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_text}]})
    messages.append({
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ],
    })
    return processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=DTYPE)


def medgemma_self_test():
    """Short generation on a synthetic image to catch overflow early."""
    test_img = Image.new("RGB", (512, 512), (60, 60, 60))
    inputs = build_inputs(test_img, "Describe this image in one sentence.")
    n = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    text = processor.decode(out[0][n:], skip_special_tokens=True).strip()
    if not text:
        raise RuntimeError(
            f"MedGemma self-test produced no text (PRECISION = '{precision_tag}'). "
            "On a T4 use PRECISION = '4bit-fp32'."
        )
    print("Self-test OK:", text[:100])


if globals().get("_MEDGEMMA_TESTED") != CACHE_KEY:
    print("Running MedGemma self-test...")
    medgemma_self_test()
    _MEDGEMMA_TESTED = CACHE_KEY


# ============================================================
# 7. ANALYSIS PIPELINE
# ============================================================

def image_to_data_url(img, max_side=OPENAI_MAX_SIDE):
    img = img.copy()
    if max(img.size) > max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    buf = BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")


class Job:
    """Shared, thread-safe state for one analysis run."""

    STEPS = ("mg", "gpt", "final")

    def __init__(self):
        self.lock = threading.Lock()
        self.text = {k: "" for k in self.STEPS}
        self.state = {k: "idle" for k in self.STEPS}
        self.error = {}
        self.t_start = {}
        self.t_end = {}

    def start(self, step):
        with self.lock:
            self.state[step] = "running"
            self.t_start[step] = time.time()

    def append(self, step, chunk):
        with self.lock:
            self.text[step] += chunk

    def finish(self, step, error=None):
        with self.lock:
            self.t_end[step] = time.time()
            if error:
                self.state[step] = "failed"
                self.error[step] = str(error)
            else:
                self.state[step] = "done"

    def skip(self, step):
        with self.lock:
            self.state[step] = "skipped"

    def elapsed(self, step):
        if step not in self.t_start:
            return None
        return self.t_end.get(step, time.time()) - self.t_start[step]

    def running(self, *steps):
        return any(self.state[s] == "running" for s in steps)


def medgemma_worker(job, img, max_tokens):
    job.start("mg")
    try:
        inputs = build_inputs(
            img, "Independently analyse this chest X-ray.", system_text=RADIOLOGY_PROMPT
        )
        streamer = TextIteratorStreamer(
            processor.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=600
        )
        gen_error = {}

        def _generate():
            try:
                with torch.inference_mode():
                    model.generate(
                        **inputs,
                        streamer=streamer,
                        max_new_tokens=int(max_tokens),
                        do_sample=False,
                        use_cache=True,
                        repetition_penalty=1.05,
                    )
            except Exception as e:
                gen_error["e"] = e
                streamer.end()

        t = threading.Thread(target=_generate, daemon=True)
        t.start()
        for chunk in streamer:
            job.append("mg", chunk)
        t.join()

        del inputs
        torch.cuda.empty_cache()

        if gen_error:
            raise gen_error["e"]
        if not job.text["mg"].strip():
            raise RuntimeError(
                f"MedGemma returned an empty report (PRECISION = '{precision_tag}')."
            )
        job.finish("mg")
    except Exception as e:
        job.finish("mg", e)


def openai_worker(job, data_url):
    job.start("gpt")
    try:
        stream = client.responses.create(
            model=OPENAI_MODEL,
            instructions=RADIOLOGY_PROMPT,
            input=[{
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Independently examine the original chest X-ray and "
                            "generate a complete radiology report. Do not assume "
                            "any previous model's interpretation."
                        ),
                    },
                    {"type": "input_image", "image_url": data_url, "detail": "high"},
                ],
            }],
            max_output_tokens=1800,
            stream=True,
        )
        for event in stream:
            if event.type == "response.output_text.delta":
                job.append("gpt", event.delta)
            elif event.type in ("response.failed", "error"):
                raise RuntimeError(f"Streaming error: {event}")
        if not job.text["gpt"].strip():
            raise RuntimeError("GPT-6 Astra returned an empty report.")
        job.finish("gpt")
    except Exception as e:
        job.finish("gpt", e)


def report_or_error(job, step, name):
    if job.state[step] == "done":
        return job.text[step].strip()
    if job.state[step] == "skipped":
        return f"*{name} report unavailable: disabled by user.*"
    return f"*{name} report unavailable: {job.error.get(step, 'unknown error')}*"


def save_reports(job, filename, mg_report, gpt_report, final_report):
    run_dir = os.path.join(OUTPUT_ROOT, datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(run_dir, exist_ok=True)
    stamp = datetime.now().strftime("%d %b %Y, %H:%M")

    full = f"""# Multi-Model Chest X-Ray Analysis

| Field | Value |
|---|---|
| Generated | {stamp} |
| Source image | {filename} |
| Model 1 | MedGemma 1.5 4B IT ({precision_tag}) |
| Model 2 | {OPENAI_MODEL} |
| Reviewer | {OPENAI_MODEL} (consensus & verification) |

> AI-assisted research report. Must be reviewed by a qualified radiologist before any clinical use.

---

{final_report}

---

# Appendix A - MedGemma Independent Report

{mg_report}

---

# Appendix B - GPT-6 Astra Independent Report

{gpt_report}
"""
    files = {
        "00_Full_Analysis.md": full,
        "01_MedGemma_Report.md": mg_report,
        "02_GPT6_Astra_Report.md": gpt_report,
        "03_Final_Verified_Report.md": final_report,
    }
    paths = []
    for name, content in files.items():
        p = os.path.join(run_dir, name)
        with open(p, "w", encoding="utf-8") as f:
            f.write(content)
        paths.append(p)
    return paths


# ============================================================
# 8. UI HELPERS
# ============================================================

STEP_META = {
    "mg":    ("Model 1", "MedGemma 1.5 4B", "Independent read · local GPU"),
    "gpt":   ("Model 2", "GPT-6 Astra", "Independent read · API"),
    "final": ("Model 3", "Consensus review", "Verification · GPT-6 Astra"),
}
STATE_LABEL = {
    "idle": "Waiting", "running": "Running", "done": "Complete",
    "failed": "Failed", "skipped": "Skipped",
}


def pipeline_html(job=None, note=""):
    cards = []
    for step, (tag, title, sub) in STEP_META.items():
        state = job.state[step] if job else "idle"
        label = STATE_LABEL[state]
        el = job.elapsed(step) if job else None
        if el is not None and state in ("running", "done", "failed"):
            label += f" · {el:.0f}s"
        cards.append(f"""
        <div class="step {state}">
          <div class="step-top"><span class="step-tag">{tag}</span>
            <span class="pill {state}"><span class="dot"></span>{label}</span></div>
          <div class="step-title">{title}</div>
          <div class="step-sub">{sub}</div>
        </div>""")
    note_html = f'<div class="pipeline-note">{note}</div>' if note else ""
    return f'<div class="pipeline">{"".join(cards)}</div>{note_html}'


def live(text, running):
    return text + (" ▌" if running else "")


def nochange():
    return gr.skip() if hasattr(gr, "skip") else gr.update()


PLACEHOLDER = "<div class='empty-state'>Upload a chest X-ray and press <b>Run analysis</b>.</div>"
WAITING_FINAL = (
    "<div class='empty-state'>The consensus review starts once both independent "
    "reads are complete.</div>"
)


# ============================================================
# 9. MAIN HANDLER (generator -> live UI updates)
# ============================================================

def analyze(image_path, use_medgemma, mg_tokens):
    if not image_path:
        raise gr.Error("Please upload a chest X-ray image first.")

    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        raise gr.Error(f"Could not read the image: {e}")

    filename = os.path.basename(image_path)
    data_url = image_to_data_url(img)
    job = Job()
    if not use_medgemma:
        job.skip("mg")

    def frame(note, tabs=None, files=None):
        return (
            pipeline_html(job, note),
            live(job.text["mg"], job.state["mg"] == "running") or PLACEHOLDER
                if job.state["mg"] != "skipped" else "*MedGemma disabled for this run.*",
            live(job.text["gpt"], job.state["gpt"] == "running") or PLACEHOLDER,
            live(job.text["final"], job.state["final"] == "running") or WAITING_FINAL,
            tabs if tabs is not None else nochange(),
            files if files is not None else nochange(),
        )

    # ---- Stage 1: independent reads in parallel ----
    executor = ThreadPoolExecutor(max_workers=2)
    if use_medgemma:
        executor.submit(medgemma_worker, job, img, mg_tokens)
    executor.submit(openai_worker, job, data_url)
    executor.shutdown(wait=False)

    yield frame(
        "Running independent reads in parallel...",
        tabs=gr.Tabs(selected="tab_mg" if use_medgemma else "tab_gpt"),
        files=None,
    )

    time.sleep(0.3)
    while job.running("mg", "gpt") or job.state["gpt"] == "idle" or (
        use_medgemma and job.state["mg"] == "idle"
    ):
        yield frame("Running independent reads in parallel...")
        time.sleep(0.4)

    mg_report = report_or_error(job, "mg", "MedGemma")
    gpt_report = report_or_error(job, "gpt", "GPT-6 Astra")

    # ---- Stage 2: consensus & verification (streamed) ----
    job.start("final")
    yield frame("Consensus review in progress...", tabs=gr.Tabs(selected="tab_final"))

    try:
        stream = client.responses.create(
            model=OPENAI_MODEL,
            instructions=VERIFICATION_PROMPT,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": build_verification_input(mg_report, gpt_report)},
                    {"type": "input_image", "image_url": data_url, "detail": "high"},
                ],
            }],
            max_output_tokens=3000,
            stream=True,
        )
        last = 0.0
        for event in stream:
            if event.type == "response.output_text.delta":
                job.append("final", event.delta)
                if time.time() - last > 0.25:
                    yield frame("Consensus review in progress...")
                    last = time.time()
            elif event.type in ("response.failed", "error"):
                raise RuntimeError(f"Streaming error: {event}")
        if not job.text["final"].strip():
            raise RuntimeError("Final verification returned an empty report.")
        job.finish("final")
    except Exception as e:
        job.finish("final", e)
        job.text["final"] += f"\n\n> **Verification error:** {e}"

    final_report = job.text["final"].strip()

    # ---- Stage 3: save & offer downloads ----
    paths = save_reports(job, filename, mg_report, gpt_report, final_report)
    total = sum(job.elapsed(s) or 0 for s in ("final",)) + max(
        job.elapsed("mg") or 0, job.elapsed("gpt") or 0
    )
    failed = [STEP_META[s][1] for s in Job.STEPS if job.state[s] == "failed"]
    note = (
        f"Analysis complete in {total:.0f}s. Reports ready for download."
        if not failed
        else f"Finished with issues ({', '.join(failed)} failed). Partial reports saved."
    )
    yield frame(note, files=paths)


def clear_all():
    return (
        None,
        pipeline_html(),
        PLACEHOLDER,
        PLACEHOLDER,
        PLACEHOLDER,
        gr.Tabs(selected="tab_final"),
        None,
        "",
    )


def image_info(image_path):
    if not image_path:
        return ""
    try:
        with Image.open(image_path) as im:
            w, h = im.size
            mode = im.mode
        size_kb = os.path.getsize(image_path) / 1024
        return (
            f"<div class='img-meta'><span><b>{w} × {h}</b> px</span>"
            f"<span>{mode}</span><span>{size_kb:,.0f} KB</span></div>"
        )
    except Exception:
        return ""


# ============================================================
# 10. THEME & STYLES
# ============================================================

THEME = gr.themes.Soft(
    primary_hue=gr.themes.colors.teal,
    secondary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(
    button_primary_background_fill="#0f766e",
    button_primary_background_fill_hover="#115e59",
    button_primary_text_color="#ffffff",
    block_radius="12px",
)

CSS = """
.gradio-container { max-width: 1380px !important; margin: 0 auto !important; }

/* ---------- header ---------- */
.app-header {
  background: linear-gradient(135deg, #0b2545 0%, #13315c 55%, #0f766e 100%);
  border-radius: 14px; padding: 22px 28px; color: #fff;
  display: flex; justify-content: space-between; align-items: center; gap: 16px;
  flex-wrap: wrap;
}
.app-header .brand { display: flex; align-items: center; gap: 14px; }
.app-header .logo {
  width: 46px; height: 46px; border-radius: 12px; background: rgba(255,255,255,.12);
  display: grid; place-items: center; font-size: 24px;
}
.app-header h1 { margin: 0; font-size: 1.45rem; font-weight: 650; color: #fff; letter-spacing: -.01em; }
.app-header p { margin: 4px 0 0; color: #cbd5e1; font-size: .92rem; }
.badges { display: flex; gap: 8px; flex-wrap: wrap; }
.badge {
  font-size: .75rem; padding: 5px 10px; border-radius: 999px;
  background: rgba(255,255,255,.12); border: 1px solid rgba(255,255,255,.2); color: #e2e8f0;
}
.badge.warn { background: rgba(245,158,11,.18); border-color: rgba(245,158,11,.45); color: #fde68a; }

/* ---------- section labels ---------- */
.section-label {
  font-size: .72rem; font-weight: 700; letter-spacing: .08em; text-transform: uppercase;
  color: var(--body-text-color-subdued); margin: 4px 2px 6px;
}

/* ---------- pipeline status ---------- */
.pipeline { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; }
@media (max-width: 820px) { .pipeline { grid-template-columns: 1fr; } }
.step {
  border: 1px solid var(--border-color-primary); border-radius: 12px;
  padding: 12px 14px; background: var(--background-fill-secondary); transition: all .2s;
}
.step.running { border-color: #14b8a6; box-shadow: 0 0 0 3px rgba(20,184,166,.15); }
.step.failed { border-color: #ef4444; }
.step-top { display: flex; justify-content: space-between; align-items: center; }
.step-tag { font-size: .7rem; font-weight: 700; letter-spacing: .06em; text-transform: uppercase;
  color: var(--body-text-color-subdued); }
.step-title { font-weight: 650; margin-top: 6px; font-size: .98rem; }
.step-sub { font-size: .78rem; color: var(--body-text-color-subdued); margin-top: 2px; }
.pill { font-size: .72rem; display: inline-flex; align-items: center; gap: 6px;
  padding: 3px 9px; border-radius: 999px; background: var(--background-fill-primary);
  border: 1px solid var(--border-color-primary); }
.pill .dot { width: 7px; height: 7px; border-radius: 50%; background: #94a3b8; }
.pill.running .dot { background: #14b8a6; animation: pulse 1.1s ease-in-out infinite; }
.pill.done .dot { background: #16a34a; }
.pill.failed .dot { background: #dc2626; }
.pill.skipped .dot { background: #cbd5e1; }
@keyframes pulse { 0%,100% { opacity: 1; transform: scale(1);} 50% { opacity: .35; transform: scale(.8);} }
.pipeline-note { margin-top: 10px; font-size: .85rem; color: var(--body-text-color-subdued); }

/* ---------- report panels ---------- */
.report-panel {
  min-height: 420px; max-height: 680px; overflow-y: auto !important;
  padding: 20px 26px !important; border: 1px solid var(--border-color-primary) !important;
  border-radius: 12px !important; background: var(--background-fill-primary) !important;
  line-height: 1.6;
}
.report-panel h1 { font-size: 1.22rem !important; font-weight: 700; padding-bottom: 8px;
  border-bottom: 2px solid #0f766e; margin: 0 0 14px !important; }
.report-panel h2 { font-size: 1.05rem !important; color: #0f766e; margin: 1.3em 0 .5em !important; }
.report-panel h3 { font-size: .96rem !important; margin: 1.1em 0 .4em !important; }
.report-panel table { font-size: .88rem; }
.report-panel blockquote { border-left: 3px solid #f59e0b; padding-left: 12px; }
.empty-state { color: var(--body-text-color-subdued); text-align: center; padding: 120px 20px; font-size: .95rem; }

/* ---------- misc ---------- */
.img-meta { display: flex; gap: 14px; font-size: .8rem; color: var(--body-text-color-subdued); padding: 2px 4px; }
.disclaimer {
  border-left: 4px solid #f59e0b; background: rgba(245,158,11,.08);
  padding: 10px 14px; border-radius: 8px; font-size: .8rem; line-height: 1.5;
}
.footer { text-align: center; font-size: .78rem; color: var(--body-text-color-subdued); padding: 8px; }
"""

HEADER_HTML = f"""
<div class="app-header">
  <div class="brand">
    <div class="logo">🩻</div>
    <div>
      <h1>Multi-Model Chest X-Ray Analysis</h1>
      <p>Independent AI reads with consensus verification · structured radiology reporting</p>
    </div>
  </div>
  <div class="badges">
    <span class="badge">MedGemma 1.5 4B · {precision_tag}</span>
    <span class="badge">{OPENAI_MODEL}</span>
    <span class="badge">{gpu_name}</span>
    <span class="badge warn">Research use only</span>
  </div>
</div>
"""

DISCLAIMER_HTML = """
<div class="disclaimer">
  <b>Not for clinical use.</b> Reports are generated by AI models for research and
  education. They may contain errors and must be reviewed by a qualified radiologist.
  Do not upload identifiable patient data.
</div>
"""


# ============================================================
# 11. BUILD THE APP
# ============================================================

GRADIO_MAJOR = int(gr.__version__.split(".")[0])
blocks_kwargs = dict(title="Chest X-Ray Analysis")
style_kwargs = dict(theme=THEME, css=CSS)
if GRADIO_MAJOR < 6:            # Gradio 6 moved theme/css to launch()
    blocks_kwargs.update(style_kwargs)
    style_kwargs = {}

with gr.Blocks(**blocks_kwargs) as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row(equal_height=False):

        # ---------------- left: input ----------------
        with gr.Column(scale=4, min_width=340):
            gr.HTML("<div class='section-label'>Study</div>")
            image_in = gr.Image(
                type="filepath",
                label="Chest X-ray (PNG / JPG)",
                sources=["upload", "clipboard"],
                height=430,
            )
            img_meta = gr.HTML()

            with gr.Accordion("Analysis settings", open=False):
                use_mg = gr.Checkbox(
                    value=True,
                    label="Include MedGemma (local GPU read)",
                    info="Turn off for a faster GPT-only run.",
                )
                mg_tokens = gr.Slider(
                    400, 1600, value=MEDGEMMA_MAX_NEW_TOKENS, step=100,
                    label="MedGemma max report length (tokens)",
                    info="Lower = faster on T4.",
                )

            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)
                run_btn = gr.Button("Run analysis", variant="primary", scale=2)

            gr.HTML(DISCLAIMER_HTML)

        # ---------------- right: results ----------------
        with gr.Column(scale=7, min_width=480):
            gr.HTML("<div class='section-label'>Pipeline</div>")
            status = gr.HTML(pipeline_html())

            gr.HTML("<div class='section-label'>Reports</div>")
            with gr.Tabs(selected="tab_final") as tabs:
                with gr.Tab("Consolidated report", id="tab_final"):
                    final_md = gr.Markdown(PLACEHOLDER, elem_classes="report-panel")
                with gr.Tab("MedGemma", id="tab_mg"):
                    mg_md = gr.Markdown(PLACEHOLDER, elem_classes="report-panel")
                with gr.Tab("GPT-6 Astra", id="tab_gpt"):
                    gpt_md = gr.Markdown(PLACEHOLDER, elem_classes="report-panel")

            files_out = gr.File(label="Download reports (.md)", file_count="multiple")

    gr.HTML(
        "<div class='footer'>Multi-model AI-assisted radiology research tool · "
        "outputs require expert review</div>"
    )

    # ---------------- events ----------------
    image_in.change(image_info, inputs=image_in, outputs=img_meta)

    run_event = run_btn.click(
        analyze,
        inputs=[image_in, use_mg, mg_tokens],
        outputs=[status, mg_md, gpt_md, final_md, tabs, files_out],
    )

    clear_btn.click(
        clear_all,
        outputs=[image_in, status, mg_md, gpt_md, final_md, tabs, files_out, img_meta],
        cancels=[run_event],
    )


# ============================================================
# 12. LAUNCH
# ============================================================

gr.close_all()   # free the port if the cell is re-run

demo.queue(default_concurrency_limit=1, max_size=10).launch(
    share=True,
    debug=True,
    show_error=True,
    **style_kwargs,
)

Authentication successful.
MedGemma already loaded (medgemma-1.5-4b-it-4bit-fp32) - using in-memory model.
GPU memory used: 4.63 GB
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d8ddae541ada06cbc4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d8ddae541ada06cbc4.gradio.live
